# LeetCode 2058 — Find the Minimum and Maximum Number of Nodes Between Critical Points bằng Python

Link đề: https://leetcode.com/problems/find-the-minimum-and-maximum-number-of-nodes-between-critical-points/ · Độ khó: **Medium** · Chủ đề: Linked List

Skeleton LeetCode:

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def nodesBetweenCriticalPoints(self, head: Optional[ListNode]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Duyệt danh sách liên kết với **cửa sổ ba nút** `(trước, hiện tại, sau)` để nhận diện **cực đại/cực tiểu địa phương** (so sánh **nghiêm ngặt**).
2. Nhận ra: khoảng cách **lớn nhất** giữa hai điểm tới hạn = (cuối − đầu); khoảng cách **nhỏ nhất** luôn đạt được giữa hai điểm tới hạn **liên tiếp**.
3. Giải trong **một lượt** `O(n)`, `O(1)` bộ nhớ: chỉ cần nhớ vị trí điểm tới hạn đầu tiên và gần nhất.
4. Xử lý trường hợp có ít hơn hai điểm tới hạn (`[-1, -1]`) và các giá trị **bằng nhau** (không phải cực trị).

## 2. Đề bài & ví dụ

**Điểm tới hạn** của danh sách liên kết là nút **cực đại địa phương** (giá trị **lớn hơn hẳn** cả nút trước và nút sau) hoặc **cực tiểu địa phương** (**nhỏ hơn hẳn** cả hai). Nút chỉ có thể là điểm tới hạn nếu có cả nút trước và nút sau.

Trả về `[minDistance, maxDistance]`: khoảng cách nhỏ nhất và lớn nhất giữa **hai điểm tới hạn khác nhau** (khoảng cách = hiệu vị trí). Nếu có ít hơn hai điểm tới hạn, trả `[-1, -1]`.

**Ví dụ 1:** `[3,1]` → `[-1,-1]`.

**Ví dụ 2:** `[5,3,1,2,5,1,2]` → `[1,3]` (điểm tới hạn ở vị trí 3, 5, 6 tính từ 1: `1` cực tiểu, `5` cực đại, `1` cực tiểu).

**Ví dụ 3:** `[1,3,2,2,3,2,2,2,7]` → `[3,3]` (chỉ vị trí 2 và 5).

**Ràng buộc:** số nút trong `[2, 10⁵]`; `1 ≤ Node.val ≤ 10⁵`.

In [1]:
import random
import time
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):
    dummy = ListNode(); cur = dummy
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

examples = [([3, 1], [-1, -1]), ([5, 3, 1, 2, 5, 1, 2], [1, 3]), ([1, 3, 2, 2, 3, 2, 2, 2, 7], [3, 3])]
for vals, exp in examples:
    print(f"{vals} -> kỳ vọng {exp}")

[3, 1] -> kỳ vọng [-1, -1]
[5, 3, 1, 2, 5, 1, 2] -> kỳ vọng [1, 3]
[1, 3, 2, 2, 3, 2, 2, 2, 7] -> kỳ vọng [3, 3]


## 3. Kiến thức nền

### 3.1 Cực trị địa phương nghiêm ngặt

`b` (giữa `a` và `c`) là cực đại nếu `b > a` và `b > c`; cực tiểu nếu `b < a` và `b < c`. Gộp lại: `(b − a)·(b − c) > 0`. Nếu có dấu bằng (`b = a` hoặc `b = c`) thì **không** phải điểm tới hạn.

In [2]:
def is_critical(a, b, c):
    return (b > a and b > c) or (b < a and b < c)

for t in [(1, 3, 2), (3, 1, 2), (2, 2, 3), (1, 2, 3), (3, 2, 2)]:
    print(t, "->", is_critical(*t), "| tích:", (t[1] - t[0]) * (t[1] - t[2]))

(1, 3, 2) -> True | tích: 2
(3, 1, 2) -> True | tích: 2
(2, 2, 3) -> False | tích: 0
(1, 2, 3) -> False | tích: -1
(3, 2, 2) -> False | tích: 0


### 3.2 Duyệt danh sách liên kết với ba con trỏ

Giữ `prev`, `cur = prev.next`, và xét `cur.next`. Đếm chỉ số `i` của `cur` để tính khoảng cách.

### 3.3 Khoảng cách nhỏ nhất trong một tập điểm đã sắp

Với các vị trí `p₁ < p₂ < … < p_k`, `min |p_i − p_j| = min (p_{t+1} − p_t)` — cặp gần nhất luôn là hai phần tử **kề nhau** trong thứ tự sắp. Khoảng cách lớn nhất là `p_k − p₁`.

In [3]:
p = [3, 5, 6, 11]
print("mọi cặp:", min(b - a for i, a in enumerate(p) for b in p[i + 1:]), "| kề nhau:", min(b - a for a, b in zip(p, p[1:])), "| lớn nhất:", p[-1] - p[0])

mọi cặp: 1 | kề nhau: 1 | lớn nhất: 8


## 4. Ý tưởng nền tảng / trực giác

Liệt kê vị trí các điểm tới hạn `p₁ < … < p_k` (theo thứ tự duyệt — tự động tăng dần). Khi `k ≥ 2`:
- `maxDistance = p_k − p₁`,
- `minDistance = min (p_{t+1} − p_t)`.

Tính tay **Ví dụ 2** `[5,3,1,2,5,1,2]` (chỉ số từ 0):

| i | bộ ba | loại |
|---|---|---|
| 1 | (5, 3, 1) | không |
| 2 | (3, 1, 2) | cực tiểu |
| 3 | (1, 2, 5) | không |
| 4 | (2, 5, 1) | cực đại |
| 5 | (5, 1, 2) | cực tiểu |

`p = [2, 4, 5]` ⇒ `min = min(2, 1) = 1`, `max = 5 − 2 = 3`.

In [4]:
vals = [5, 3, 1, 2, 5, 1, 2]
crit = [i for i in range(1, len(vals) - 1) if is_critical(vals[i - 1], vals[i], vals[i + 1])]
print("điểm tới hạn:", crit, "| min:", min(b - a for a, b in zip(crit, crit[1:])), "| max:", crit[-1] - crit[0])

điểm tới hạn: [2, 4, 5] | min: 1 | max: 3


## 5. Cách 1 — Brute Force / baseline: chuyển sang mảng, xét mọi cặp điểm tới hạn

In [5]:
def to_list(head):
    out = []
    while head: out.append(head.val); head = head.next
    return out

def critical_bruteforce(head):
    v = to_list(head)
    crit = [i for i in range(1, len(v) - 1) if is_critical(v[i - 1], v[i], v[i + 1])]
    if len(crit) < 2:
        return [-1, -1]
    dists = [b - a for i, a in enumerate(crit) for b in crit[i + 1:]]
    return [min(dists), max(dists)]

for vals, exp in examples:
    got = critical_bruteforce(build(vals)); print(got, "kỳ vọng", exp); assert got == exp

[-1, -1] kỳ vọng [-1, -1]
[1, 3] kỳ vọng [1, 3]
[3, 3] kỳ vọng [3, 3]


**Độ phức tạp:** `O(n + k²)` với `k` = số điểm tới hạn (tới `n − 2`, ví dụ dãy răng cưa `1,2,1,2,…`) ⇒ `O(n²)` tệ nhất; bộ nhớ `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** min chỉ cần xét các cặp **liên tiếp** (§3.3); max chỉ cần điểm **đầu** và **cuối**.

**Insight 2 (một lượt):** khi gặp điểm tới hạn ở vị trí `i`: nếu đã có điểm trước `last`, cập nhật `best_min = min(best_min, i − last)`; nếu chưa có `first`, đặt `first = i`; luôn đặt `last = i`. Kết thúc: nếu `first == last` (hoặc chưa có) ⇒ `[-1, -1]`, ngược lại `[best_min, last − first]`.

**Invariant:** sau khi xét vị trí `i`, `first`/`last` là điểm tới hạn đầu/cuối trong `[1, i]`, `best_min` là khoảng cách nhỏ nhất giữa hai điểm liên tiếp trong số đó.

### 6.1 Minh họa: phân bố điểm tới hạn trên các dạng danh sách

Ba dạng điển hình: **răng cưa** (mọi nút giữa đều là điểm tới hạn ⇒ `min = 1`), **núi** (tăng rồi giảm ⇒ đúng một đỉnh ⇒ `[-1,-1]`), và **sóng có bậc phẳng** (dấu bằng phá vỡ cực trị). In vị trí điểm tới hạn và đáp án của từng dạng để thấy công thức "cặp kề / đầu–cuối" hoạt động.

In [6]:
shapes = {
    "răng cưa": [1, 3, 1, 3, 1, 3, 1],
    "núi": [1, 2, 3, 4, 3, 2, 1],
    "sóng có bậc": [1, 3, 3, 1, 1, 4, 2, 2, 5, 1],
    "hai đỉnh xa": [1, 5, 1, 1, 1, 1, 1, 1, 5, 1],
}
for name, v in shapes.items():
    crit = [i for i in range(1, len(v) - 1) if is_critical(v[i - 1], v[i], v[i + 1])]
    print(f"{name:<12} {v}: điểm tới hạn {crit} -> {critical_bruteforce(build(v))}")

răng cưa     [1, 3, 1, 3, 1, 3, 1]: điểm tới hạn [1, 2, 3, 4, 5] -> [1, 4]
núi          [1, 2, 3, 4, 3, 2, 1]: điểm tới hạn [3] -> [-1, -1]
sóng có bậc  [1, 3, 3, 1, 1, 4, 2, 2, 5, 1]: điểm tới hạn [5, 8] -> [3, 3]
hai đỉnh xa  [1, 5, 1, 1, 1, 1, 1, 1, 5, 1]: điểm tới hạn [1, 8] -> [7, 7]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng giá trị + các cặp liên tiếp

In [7]:
def critical_array(head):
    v = to_list(head)
    crit = [i for i in range(1, len(v) - 1) if (v[i] - v[i - 1]) * (v[i] - v[i + 1]) > 0]
    if len(crit) < 2:
        return [-1, -1]
    return [min(b - a for a, b in zip(crit, crit[1:])), crit[-1] - crit[0]]

print([critical_array(build(v)) for v, _ in examples])

[[-1, -1], [1, 3], [3, 3]]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Một lượt trên danh sách liên kết (bản nộp)

In [8]:
class Solution:
    def nodesBetweenCriticalPoints(self, head: Optional[ListNode]) -> List[int]:
        first = last = -1
        best = float("inf")
        prev, cur, i = head, head.next, 1
        while cur and cur.next:
            a, b, c = prev.val, cur.val, cur.next.val
            if (b > a and b > c) or (b < a and b < c):
                if last != -1:
                    best = min(best, i - last)
                else:
                    first = i
                last = i
            prev, cur, i = cur, cur.next, i + 1
        if first == last:                         # 0 hoặc 1 điểm tới hạn
            return [-1, -1]
        return [best, last - first]

print([Solution().nodesBetweenCriticalPoints(build(v)) for v, _ in examples])

[[-1, -1], [1, 3], [3, 3]]


Khi không có điểm nào, `first = last = −1`; khi có đúng một, `first = last = i` ⇒ cả hai trường hợp `first == last`.

**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Dùng `zip` trên ba bản dịch của mảng

Kiểu Python: `zip(v, v[1:], v[2:])` sinh mọi bộ ba liên tiếp; `enumerate(..., 1)` cho chỉ số nút giữa.

In [9]:
def critical_zip(head):
    v = to_list(head)
    crit = [i for i, (a, b, c) in enumerate(zip(v, v[1:], v[2:]), 1) if (b - a) * (b - c) > 0]
    return [-1, -1] if len(crit) < 2 else [min(y - x for x, y in zip(crit, crit[1:])), crit[-1] - crit[0]]

print([critical_zip(build(v)) for v, _ in examples])

[[-1, -1], [1, 3], [3, 3]]


## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 3** `[1,3,2,2,3,2,2,2,7]`:

In [10]:
def trace(vals):
    head = build(vals); first = last = -1; best = float("inf")
    prev, cur, i = head, head.next, 1
    while cur and cur.next:
        a, b, c = prev.val, cur.val, cur.next.val
        crit = (b > a and b > c) or (b < a and b < c)
        if crit:
            if last != -1: best = min(best, i - last)
            else: first = i
            last = i
        print(f"i={i}: ({a},{b},{c}) {'TỚI HẠN' if crit else '-':<8} first={first}, last={last}, best={best}")
        prev, cur, i = cur, cur.next, i + 1
    return [-1, -1] if first == last else [best, last - first]

assert trace([1, 3, 2, 2, 3, 2, 2, 2, 7]) == [3, 3]

i=1: (1,3,2) TỚI HẠN  first=1, last=1, best=inf
i=2: (3,2,2) -        first=1, last=1, best=inf
i=3: (2,2,3) -        first=1, last=1, best=inf
i=4: (2,3,2) TỚI HẠN  first=1, last=4, best=3
i=5: (3,2,2) -        first=1, last=4, best=3
i=6: (2,2,2) -        first=1, last=4, best=3
i=7: (2,2,7) -        first=1, last=4, best=3


| i | bộ ba | tới hạn? | first | last | best |
|---|---|---|---|---|---|
| 1 | (1,3,2) | cực đại | 1 | 1 | ∞ |
| 2 | (3,2,2) | không (bằng nhau) | 1 | 1 | ∞ |
| 3 | (2,2,3) | không | 1 | 1 | ∞ |
| 4 | (2,3,2) | cực đại | 1 | 4 | 3 |
| 5–7 | có dấu bằng / tăng | không | 1 | 4 | 3 |

Kết quả `[3, 4 − 1] = [3, 3]` ✓. Chú ý nút cuối `7` không xét vì không có nút sau.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — một lượt, `O(1)` bộ nhớ.

In [11]:
class Solution:
    def nodesBetweenCriticalPoints(self, head: Optional[ListNode]) -> List[int]:
        first = last = -1
        best = float("inf")
        prev, cur, i = head, head.next, 1
        while cur and cur.next:
            a, b, c = prev.val, cur.val, cur.next.val
            if (b > a and b > c) or (b < a and b < c):
                if last != -1:
                    best = min(best, i - last)
                else:
                    first = i
                last = i
            prev, cur, i = cur, cur.next, i + 1
        if first == last:
            return [-1, -1]
        return [best, last - first]

for vals, exp in examples:
    got = Solution().nodesBetweenCriticalPoints(build(vals)); print(got, exp); assert got == exp

[-1, -1] [-1, -1]
[1, 3] [1, 3]
[3, 3] [3, 3]


## 10. Vì sao thuật toán đúng

**Nhận diện.** Vòng lặp xét đúng mọi nút có cả nút trước và sau (chỉ số `1..n−2`), với điều kiện nghiêm ngặt của đề.

**Max.** Với các vị trí tới hạn tăng dần `p₁ < … < p_k`, mọi hiệu `p_j − p_i ≤ p_k − p₁` và đạt được ⇒ max = `last − first`.

**Min.** Nếu `i < j` không kề nhau, có `t` với `p_i < p_t < p_j` ⇒ `p_j − p_i > p_t − p_i ≥ min các hiệu kề` ⇒ min đạt ở một cặp kề. Mỗi lần gặp điểm mới `p_{t+1}`, ta so `p_{t+1} − p_t` (`last` là `p_t`) ⇒ `best` = min các hiệu kề.

**Ít hơn hai điểm.** `first == last` ⇔ `k ≤ 1` ⇒ `[-1, -1]`. ∎

### 10.1 Kiểm chứng thực nghiệm

So bốn cách trên danh sách ngẫu nhiên giá trị nhỏ (nhiều giá trị bằng nhau), và kiểm tra `min ≤ max`, `min ≥ 1`.

In [12]:
rng = random.Random(1010)
for _ in range(3000):
    vals = [rng.randint(1, rng.choice([2, 3, 10])) for _ in range(rng.randint(2, 15))]
    ref = critical_bruteforce(build(vals))
    assert ref == critical_array(build(vals)) == critical_zip(build(vals)) == Solution().nodesBetweenCriticalPoints(build(vals))
    if ref != [-1, -1]: assert 1 <= ref[0] <= ref[1]
print("4 cách khớp nhau trên 3000 danh sách; min ≥ 1 và min ≤ max")

4 cách khớp nhau trên 3000 danh sách; min ≥ 1 và min ≤ max


### 10.2 Kiểm chứng mệnh đề "cặp gần nhất là cặp kề"

Với tập vị trí ngẫu nhiên đã sắp, so `min` qua **mọi cặp** với `min` qua **cặp kề**; đồng thời kiểm tra `max` qua mọi cặp bằng `cuối − đầu`. Đây là bổ đề then chốt cho phép giải trong một lượt.

In [13]:
rng = random.Random(102)
for _ in range(2000):
    p = sorted(rng.sample(range(100), rng.randint(2, 12)))
    allpairs = [b - a for i, a in enumerate(p) for b in p[i + 1:]]
    assert min(allpairs) == min(b - a for a, b in zip(p, p[1:])) and max(allpairs) == p[-1] - p[0]
print("min mọi cặp = min cặp kề, max mọi cặp = cuối − đầu (2000 tập)")

min mọi cặp = min cặp kề, max mọi cặp = cuối − đầu (2000 tập)


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n + k²)` | `O(n)` |
| 2. Mảng + cặp kề | `O(n)` | `O(n)` |
| 3. Một lượt | `O(n)` | `O(1)` |
| 4. `zip` | `O(n)` | `O(n)` |

In [14]:
zigzag = [1 + (i % 2) for i in range(10 ** 5)]          # mọi nút giữa đều là điểm tới hạn
for name, f in [("array", critical_array), ("zip", critical_zip), ("submit", Solution().nodesBetweenCriticalPoints)]:
    h = build(zigzag); t = time.perf_counter(); r = f(h); print(f"{name:<7}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")
h = build(zigzag[:3000]); t = time.perf_counter(); critical_bruteforce(h); print(f"brute n = 3000 (k ≈ 3000): {(time.perf_counter() - t) * 1000:.0f} ms")

array  : [1, 99997] (15 ms)


zip    : [1, 99997] (13 ms)


submit : [1, 99997] (11 ms)


brute n = 3000 (k ≈ 3000): 228 ms


## 12. Bộ test

In [15]:
ALL = {"brute": critical_bruteforce, "array": critical_array, "zip": critical_zip, "submit": Solution().nodesBetweenCriticalPoints}
def run_all(vals):
    res = {k: f(build(vals)) for k, f in ALL.items()}
    vs = list(res.values()); assert all(v == vs[0] for v in vs), (vals, res)
    return vs[0]

for vals, exp in examples:
    assert run_all(vals) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [([1, 2], [-1, -1]), ([1, 2, 1], [-1, -1]), ([1, 2, 1, 2], [1, 1]), ([2, 2, 2, 2], [-1, -1]),
        ([1, 3, 1, 1, 1, 3, 1], [4, 4]), ([1, 2, 3, 4], [-1, -1]), ([3, 1, 3, 1, 3], [1, 2])]
for vals, exp in edge:
    assert run_all(vals) == exp, (vals, run_all(vals))
rng = random.Random(20580)
for _ in range(500):
    run_all([rng.randint(1, rng.choice([2, 5, 10 ** 5])) for _ in range(rng.randint(2, 30))])
print("Ví dụ, 7 edge case, 500 danh sách ngẫu nhiên: OK")

Ví dụ, 7 edge case, 500 danh sách ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- Đảo ngược danh sách không đổi đáp án.
- Cộng một hằng số vào mọi nút không đổi đáp án; nhân `−1` (đổi cực đại ↔ cực tiểu, dùng `C − val`) không đổi đáp án.
- Dãy đơn điệu (tăng hoặc giảm) luôn cho `[-1, -1]`.

In [16]:
rng = random.Random(121)
f = Solution().nodesBetweenCriticalPoints
for _ in range(1000):
    v = [rng.randint(1, 6) for _ in range(rng.randint(2, 20))]; a = f(build(v))
    assert f(build(v[::-1])) == a and f(build([x + 7 for x in v])) == a and f(build([100 - x for x in v])) == a
    assert f(build(sorted(v))) == [-1, -1] and f(build(sorted(v, reverse=True))) == [-1, -1]
print("3 tính chất đúng trên 1000 danh sách")

3 tính chất đúng trên 1000 danh sách


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh không nghiêm ngặt (`>=`, `<=`)

In [17]:
def wrong_nonstrict(head):
    v = to_list(head)
    crit = [i for i in range(1, len(v) - 1) if (v[i] >= v[i - 1] and v[i] >= v[i + 1]) or (v[i] <= v[i - 1] and v[i] <= v[i + 1])]   # SAI
    return [-1, -1] if len(crit) < 2 else [min(b - a for a, b in zip(crit, crit[1:])), crit[-1] - crit[0]]

for vals, exp in examples:
    print("sai :", wrong_nonstrict(build(vals)), "| đúng:", exp)

sai : [-1, -1] | đúng: [-1, -1]
sai : [1, 3] | đúng: [1, 3]
sai : [1, 6] | đúng: [3, 3]


**Vì sao sai:** đề yêu cầu **lớn hơn hẳn / nhỏ hơn hẳn**; các nút trên "bậc bằng phẳng" (`2,2,2`) bị nhận nhầm. **Sửa:** `>` và `<`.

### Lỗi 2 — Lấy min là hiệu giữa điểm đầu và điểm thứ hai

In [18]:
def wrong_min_first_two(head):
    v = to_list(head)
    crit = [i for i in range(1, len(v) - 1) if is_critical(v[i - 1], v[i], v[i + 1])]
    return [-1, -1] if len(crit) < 2 else [crit[1] - crit[0], crit[-1] - crit[0]]   # SAI

for vals, exp in examples:
    print("sai :", wrong_min_first_two(build(vals)), "| đúng:", exp)

sai : [-1, -1] | đúng: [-1, -1]
sai : [2, 3] | đúng: [1, 3]
sai : [3, 3] | đúng: [3, 3]


**Vì sao sai:** cặp gần nhất có thể ở bất kỳ đâu (Ví dụ 2: cặp vị trí 4, 5). **Sửa:** min qua **mọi** cặp kề.

### Lỗi 3 — Xét cả nút đầu/cuối

In [19]:
def wrong_ends(head):
    v = to_list(head); n = len(v)
    def crit(i):
        nb = [v[j] for j in (i - 1, i + 1) if 0 <= j < n]
        return all(v[i] > x for x in nb) or all(v[i] < x for x in nb)       # SAI: nút mép chỉ có 1 láng giềng
    c = [i for i in range(n) if crit(i)]
    return [-1, -1] if len(c) < 2 else [min(b - a for a, b in zip(c, c[1:])), c[-1] - c[0]]

for vals, exp in examples:
    print("sai :", wrong_ends(build(vals)), "| đúng:", exp)

sai : [1, 1] | đúng: [-1, -1]
sai : [1, 6] | đúng: [1, 3]
sai : [1, 8] | đúng: [3, 3]


**Vì sao sai:** đề: điểm tới hạn phải có **cả** nút trước và nút sau. Ví dụ 1 `[3,1]` bản sai coi cả hai nút là cực trị. **Sửa:** chỉ xét `1..n−2`.

### Lỗi 4 — Trả `[inf, 0]` hoặc `[-1, 0]` khi chỉ có một điểm tới hạn

In [20]:
def wrong_one_point(head):
    first = last = -1; best = float("inf"); prev, cur, i = head, head.next, 1
    while cur and cur.next:
        if is_critical(prev.val, cur.val, cur.next.val):
            if last != -1: best = min(best, i - last)
            else: first = i
            last = i
        prev, cur, i = cur, cur.next, i + 1
    if first == -1: return [-1, -1]
    return [best, last - first]                                              # SAI khi đúng 1 điểm

for vals in ([1, 2, 1], [1, 3, 2, 2]):        # tự dựng (đối chiếu lại đề gốc)
    print("sai :", wrong_one_point(build(vals)), "| đúng:", Solution().nodesBetweenCriticalPoints(build(vals)))

sai : [inf, 0] | đúng: [-1, -1]
sai : [inf, 0] | đúng: [-1, -1]


**Vì sao sai:** "ít hơn **hai** điểm" gồm cả trường hợp đúng một điểm. **Sửa:** điều kiện `first == last`.

### Lỗi 5 — Đánh chỉ số rồi quên tăng `i` (hoặc tăng hai lần)

In [21]:
def wrong_index(head):
    first = last = -1; best = float("inf"); prev, cur, i = head, head.next, 1
    while cur and cur.next:
        if is_critical(prev.val, cur.val, cur.next.val):
            if last != -1: best = min(best, i - last)
            else: first = i
            last = i
            i += 1                                                           # SAI: chỉ tăng khi gặp điểm tới hạn
        prev, cur = cur, cur.next
    return [-1, -1] if first == last else [best, last - first]

for vals, exp in examples:
    print("sai :", wrong_index(build(vals)), "| đúng:", exp)

sai : [-1, -1] | đúng: [-1, -1]
sai : [1, 2] | đúng: [1, 3]
sai : [1, 1] | đúng: [3, 3]


**Vì sao sai:** khoảng cách tính theo **vị trí nút**, không phải số thứ tự của điểm tới hạn. **Sửa:** tăng `i` ở **mỗi** bước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Min/max khoảng cách giữa các điểm thỏa điều kiện" trên dãy ⇒ chỉ cần **điểm đầu, điểm trước, điểm hiện tại**.
- Cực trị địa phương: `(b − a)·(b − c) > 0`.
- Danh sách liên kết: giữ `prev` + `cur` + `cur.next`; đếm chỉ số riêng.
- Kiểm tra trường hợp 0/1 phần tử thỏa ⇒ `-1`.

## 15. Biến thể của bài

1. **162 Find Peak Element**: tìm một cực đại bằng binary search.
2. **Đếm** số cực đại/cực tiểu, hoặc chỉ khoảng cách giữa các **cực đại**.
3. **Cực trị không nghiêm ngặt trên bậc phẳng** ("plateau"): coi cả bậc là một điểm — cần gom đoạn bằng nhau (Bài tập 5).
4. **376 Wiggle Subsequence**: số điểm đổi chiều.
5. **Mảng vòng**: nút đầu/cuối có láng giềng vòng.

### 15.1 Code biến thể: 376 Wiggle Subsequence (đếm điểm đổi chiều)

In [22]:
def wiggle_max_length(nums):
    up = down = 1
    for a, b in zip(nums, nums[1:]):
        if b > a: up = down + 1
        elif b < a: down = up + 1
    return max(up, down) if nums else 0

def wiggle_brute(nums):
    from itertools import combinations
    best = 1
    for r in range(2, len(nums) + 1):
        for c in combinations(nums, r):
            d = [y - x for x, y in zip(c, c[1:])]
            if all(x != 0 for x in d) and all(d[i] * d[i + 1] < 0 for i in range(len(d) - 1)):
                best = max(best, r)
    return best

rng = random.Random(376)
for _ in range(300):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 9))]; assert wiggle_max_length(a) == wiggle_brute(a)
print("376 khớp vét cạn;", wiggle_max_length([1, 7, 4, 9, 2, 5]), "(LeetCode: 6)")

376 khớp vét cạn; 6 (LeetCode: 6)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | Vét cạn khoảng cách | `O(n + k²)` | `O(n)` | Kiểm chứng |
| 2. Mảng + cặp kề | Hai công thức | `O(n)` | `O(n)` | Dễ đọc |
| 3. Một lượt | first/last/best | `O(n)` | `O(1)` | **Bản nộp** |
| 4. `zip` | Bộ ba | `O(n)` | `O(n)` | Code ngắn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1,2,1,2]`; (b) `[2,2,2,2]`; (c) `[1,3,1,1,1,3,1]`; (d) `[3,1,3,1,3]`.

**Bài 2 (tính tay):** Liệt kê điểm tới hạn của `[4,2,2,5,3,6,6,1,8]` và tính đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `critical_positions(vals)`.

**Bài 4 (tự viết lại):** Viết `between_maxima(head)` — như bài gốc nhưng chỉ tính các **cực đại** địa phương.

**Bài 5 (mini challenge):** Biến thể "bậc phẳng": gom các đoạn giá trị bằng nhau liên tiếp thành một khối; một khối (không ở mép) là điểm tới hạn nếu nó lớn hơn hẳn (hoặc nhỏ hơn hẳn) cả khối trước và khối sau; vị trí của khối là vị trí nút **đầu** của nó. Viết `plateau_critical(vals)` trả `[min, max]` như bài gốc; đối chiếu một cài đặt trực tiếp.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def critical_positions(vals):
    # TODO: [i for i in 1..n-2 nếu (vals[i] - vals[i-1]) * (vals[i] - vals[i+1]) > 0]
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def between_maxima(head):
    # TODO: như Cách 3 nhưng chỉ khi b > a và b > c
    pass

def plateau_critical(vals):
    # TODO: dùng itertools.groupby để có (giá trị, vị trí đầu) mỗi khối, rồi xét bộ ba khối
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: xét các nút giữa. Mức 2: nghiêm ngặt. Mức 3: (a) `[1,1]`; (b) `[-1,-1]`; (c) `[4,4]` (vị trí 1 và 5); (d) `[1,2]` (vị trí 1, 2, 3).

**Bài 2.** Mức 1: xét từng bộ ba; các bộ có dấu bằng như `(4,2,2)`, `(3,6,6)` bị loại. Mức 2: vị trí 3 `(2,5,3)` cực đại, vị trí 4 `(5,3,6)` cực tiểu, vị trí 7 `(6,1,8)` cực tiểu. Mức 3: vị trí `3, 4, 7` ⇒ `[1, 4]`.

**Bài 3.** Mức 1: duyệt `1..n−2`. Mức 2: điều kiện tích dương. Mức 3: list comprehension.

**Bài 4.** Mức 1: chỉ giữ `b > a and b > c`. Mức 2: phần còn lại giữ nguyên. Mức 3: đối chiếu bằng lọc mảng.

**Bài 5.** Mức 1: `[(k, start) for k, g in groupby(...)]` cần đếm vị trí. Mức 2: khối ở mép không xét. Mức 3: áp dụng công thức min cặp kề / max đầu-cuối trên vị trí khối.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
f = Solution().nodesBetweenCriticalPoints
b1 = [f(build(v)) for v in ([1, 2, 1, 2], [2, 2, 2, 2], [1, 3, 1, 1, 1, 3, 1], [3, 1, 3, 1, 3])]
print("Bài 1:", b1); assert b1 == [[1, 1], [-1, -1], [4, 4], [1, 2]]
def critical_positions(vals):
    return [i for i in range(1, len(vals) - 1) if (vals[i] - vals[i - 1]) * (vals[i] - vals[i + 1]) > 0]
v2 = [4, 2, 2, 5, 3, 6, 6, 1, 8]
print("Bài 2:", critical_positions(v2), f(build(v2))); assert critical_positions(v2) == [3, 4, 7] and f(build(v2)) == [1, 4]
print("Bài 3: OK")

Bài 1: [[1, 1], [-1, -1], [4, 4], [1, 2]]
Bài 2: [3, 4, 7] [1, 4]
Bài 3: OK


In [26]:
# Bài 4, 5
from itertools import groupby
def between_maxima(head):
    first = last = -1; best = float("inf"); prev, cur, i = head, head.next, 1
    while cur and cur.next:
        if cur.val > prev.val and cur.val > cur.next.val:
            if last != -1: best = min(best, i - last)
            else: first = i
            last = i
        prev, cur, i = cur, cur.next, i + 1
    return [-1, -1] if first == last else [best, last - first]

def plateau_critical(vals):
    blocks, pos = [], 0
    for k, g in groupby(vals):
        L = sum(1 for _ in g); blocks.append((k, pos)); pos += L
    crit = [blocks[t][1] for t in range(1, len(blocks) - 1)
            if (blocks[t][0] - blocks[t - 1][0]) * (blocks[t][0] - blocks[t + 1][0]) > 0]
    return [-1, -1] if len(crit) < 2 else [min(b - a for a, b in zip(crit, crit[1:])), crit[-1] - crit[0]]

def plateau_direct(vals):
    n = len(vals); crit = []
    for i in range(n):
        if i > 0 and vals[i - 1] == vals[i]: continue            # chỉ đầu khối
        j = i
        while j + 1 < n and vals[j + 1] == vals[i]: j += 1
        if i == 0 or j == n - 1: continue
        if (vals[i] > vals[i - 1] and vals[i] > vals[j + 1]) or (vals[i] < vals[i - 1] and vals[i] < vals[j + 1]):
            crit.append(i)
    return [-1, -1] if len(crit) < 2 else [min(b - a for a, b in zip(crit, crit[1:])), crit[-1] - crit[0]]

rng = random.Random(5)
for _ in range(500):
    v = [rng.randint(1, 4) for _ in range(rng.randint(2, 15))]
    mx = [i for i in range(1, len(v) - 1) if v[i] > v[i - 1] and v[i] > v[i + 1]]
    assert between_maxima(build(v)) == ([-1, -1] if len(mx) < 2 else [min(b - a for a, b in zip(mx, mx[1:])), mx[-1] - mx[0]])
    assert plateau_critical(v) == plateau_direct(v)
print("Bài 4, 5: OK |", between_maxima(build([5, 3, 1, 2, 5, 1, 2])), plateau_critical([1, 3, 2, 2, 3, 2, 2, 2, 7]))

Bài 4, 5: OK | [-1, -1] [1, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
first = last = -1; best = inf
prev, cur, i = head, head.next, 1
while cur and cur.next:
    a, b, c = prev.val, cur.val, cur.next.val
    if (b > a and b > c) or (b < a and b < c):
        if last != -1: best = min(best, i - last)
        else: first = i
        last = i
    prev, cur, i = cur, cur.next, i + 1
return [-1, -1] if first == last else [best, last - first]
```

**Checklist:**
- [ ] So sánh **nghiêm ngặt**?
- [ ] Không xét nút đầu/cuối?
- [ ] Min qua các cặp **kề**, max = cuối − đầu?
- [ ] Ít hơn **hai** điểm ⇒ `[-1, -1]` (kể cả đúng một)?
- [ ] Tăng chỉ số `i` ở mọi bước?